# 01. Echoes / FMA 데이터 품질 검증

## 프로젝트 주제
**미지 생성기와 오디오 압축 환경에서의 강건한 AI 생성 음악 탐지**

이 노트북은 모델 학습 전에 데이터셋 구조와 품질을 검증한 과정을 정리한다.

### 현재까지 확인한 핵심 결과

| 항목 | 확인 결과 |
|---|---:|
| Echoes manifest 전체 행 | 4,468 |
| TTA | 3,165 |
| ATA | 1,303 |
| 생성기 수 | 12 |
| 장르 수 | 3 |
| TTA 중복 충돌 행 | 3 |
| Clean TTA | **3,162** |
| Clean TTA original_audio 그룹 | **296** |
| Clean TTA 실제 파일 누락 | **0** |
| 실제 오디오 파일 수 | 4,488 |
| Manifest 고유 파일 경로 수 | 4,464 |
| Manifest 밖 extra audio | 24 |

연구 범위:
- REAL: FMA
- FAKE: Echoes TTA
- ATA 제외
- 장르: Electronic / Rock / Pop
- 이후 `original_audio` 기준 그룹 분할로 데이터 누수 방지


## 0. 프로젝트 경로와 데이터 구조

```text
project/
├── data/
│   ├── raw/
│   │   ├── Echoes/Echoes/
│   │   │   ├── dataset_manifest.csv
│   │   │   ├── TTA/
│   │   │   └── ATA/
│   │   └── FMA/fma_metadata/
│   │       ├── tracks.csv
│   │       ├── genres.csv
│   │       └── ...
│   ├── metadata/
│   └── processed/
├── notebooks/
├── src/
├── experiments/
├── results/
└── checkpoints/
```


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_META_ROOT = PROJECT_ROOT / "data/raw/FMA/fma_metadata"
FMA_TRACKS = FMA_META_ROOT / "tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())


## 1. Echoes manifest 기본 구조 확인

확인된 컬럼:
- `path_in_dataset`
- `original_audio`
- `generator`
- `type`
- `genre`
- `description`
- `duration`


In [ ]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("shape:", echoes.shape)
print("\ncolumns:")
print(echoes.columns.tolist())

display(echoes.head(3))


In [ ]:
print("===== TYPE =====")
print(echoes["type"].value_counts(dropna=False))

print("\n===== GENERATOR =====")
print(echoes["generator"].value_counts(dropna=False))

print("\n===== GENRE =====")
print(echoes["genre"].value_counts(dropna=False))


### 실제 확인 결과

**TYPE**
- TTA: 3,165
- ATA: 1,303

**GENRE**
- Electronic: 1,653
- Rock: 1,594
- Pop: 1,221

생성기 전체는 12종이다.


## 2. TTA만 필터링

In [ ]:
tta = echoes[echoes["type"] == "TTA"].copy()

print("TTA rows:", len(tta))
print("Unique original_audio:", tta["original_audio"].nunique())
print("Unique generators:", tta["generator"].nunique())

print("\n===== TTA GENERATOR =====")
print(tta["generator"].value_counts())

print("\n===== TTA GENRE =====")
print(tta["genre"].value_counts())

print("\n===== DURATION =====")
print(tta["duration"].describe())


## 3. TTA 중복 파일 경로 검사

MusicGen TTA에서 하나의 실제 파일이 서로 다른 원곡 3개에 연결된 충돌이 확인되었다.


In [ ]:
tta_dup = (
    tta[tta["path_in_dataset"].duplicated(keep=False)]
    .sort_values("path_in_dataset")
)

print("Duplicated TTA rows:", len(tta_dup))
print("Duplicated TTA unique paths:", tta_dup["path_in_dataset"].nunique())

display(
    tta_dup[
        ["path_in_dataset", "original_audio", "generator", "genre"]
    ]
)


### 확인된 충돌

동일 파일:

```text
TTA/musicgen/_musicgen_TTA_001.wav
```

서로 다른 원곡 3개가 같은 파일을 참조하므로 어느 원곡과 실제로 대응되는지 확정할 수 없다.  
따라서 **3행 모두 제외**한다.


## 4. Clean TTA 생성 및 실제 파일 존재 확인

In [ ]:
dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

tta_clean["full_path"] = tta_clean["path_in_dataset"].apply(
    lambda x: ECHOES_ROOT / x
)
tta_clean["exists"] = tta_clean["full_path"].apply(lambda p: p.exists())

print("===== CLEAN TTA =====")
print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Generators        :", tta_clean["generator"].nunique())
print("Original groups   :", tta_clean["original_audio"].nunique())

print("\n===== GENERATOR COUNTS =====")
print(tta_clean["generator"].value_counts())

print("\n===== GENRE COUNTS =====")
print(tta_clean["genre"].value_counts())

print("\n===== FILE CHECK =====")
print("Existing files :", int(tta_clean["exists"].sum()))
print("Missing files  :", int((~tta_clean["exists"]).sum()))


### Clean TTA 최종 결과

- 원본 TTA: **3,165**
- 중복 충돌 제외: **3**
- Clean TTA: **3,162**
- 생성기: **12종**
- `original_audio` 그룹: **296개**
- 실제 파일 존재: **3,162 / 3,162**
- Missing file: **0**

생성기별 Clean TTA:
- suno 300
- udio 300
- elevenlabs 300
- diffrhythm 299
- brev 298
- acestep 294
- musicgen 293
- audioldm 292
- songgen 292
- stableaudio 194
- producer 151
- mubert 149

장르별:
- Electronic 1,131
- Rock 1,127
- Pop 904


## 5. Manifest와 실제 오디오 파일 비교

In [ ]:
manifest_paths = set(
    echoes["path_in_dataset"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

audio_exts = {".wav", ".mp3", ".flac"}
actual_paths = set()

for p in ECHOES_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in audio_exts:
        actual_paths.add(p.relative_to(ECHOES_ROOT).as_posix())

extra_files = sorted(actual_paths - manifest_paths)
missing_files = sorted(manifest_paths - actual_paths)

print("===== MANIFEST vs ACTUAL AUDIO =====")
print("Manifest rows         :", len(echoes))
print("Unique manifest paths :", len(manifest_paths))
print("Actual audio files    :", len(actual_paths))
print("Extra audio files     :", len(extra_files))
print("Missing audio files   :", len(missing_files))

print("\n===== EXTRA FILES =====")
for x in extra_files:
    print(x)


### 확인 결과

- Manifest rows: **4,468**
- Unique manifest paths: **4,464**
- Actual audio files: **4,488**
- Extra audio files: **24**
- Missing audio files: **0**

Extra 24개:
- ATA/songgen: 23개
- TTA/acestep: 1개

Manifest에 없는 extra 파일은 metadata 연결을 보장할 수 없으므로 연구 데이터에 포함하지 않는다.


## 6. 전체 manifest 중복 경로 검사

In [ ]:
all_dup = (
    echoes[echoes["path_in_dataset"].duplicated(keep=False)]
    .sort_values("path_in_dataset")
)

print("Duplicated rows        :", len(all_dup))
print("Duplicated unique paths:", all_dup["path_in_dataset"].nunique())

display(
    all_dup[
        ["path_in_dataset", "original_audio", "generator", "type", "genre"]
    ]
)


### 전체 중복 구조

중복 경로는 2개다.

- `ATA/musicgen/_musicgen_ATA_001.wav`
- `TTA/musicgen/_musicgen_TTA_001.wav`

각 경로가 서로 다른 3개 원곡 행에서 반복되어 전체 manifest에서 중복 초과분은 4행이다.


## 7. FMA metadata 구조 확인

In [ ]:
fma = pd.read_csv(
    FMA_TRACKS,
    header=[0, 1],
    index_col=0
)

print("FMA shape:", fma.shape)
print("\nFirst 50 columns:")
print(fma.columns.tolist()[:50])

display(fma.head(3))


### 이후 사용할 FMA 주요 컬럼

- `('artist', 'name')`
- `('track', 'title')`
- `('track', 'genre_top')`
- `('track', 'license')`
- `('track', 'duration')`
- `('set', 'subset')`

FMA 전체 `tracks.csv`에는 **106,574곡**이 있다.


## 8. 현재 결론

### Echoes FAKE 데이터 정제 결과

```text
Echoes 전체 manifest 4,468
        ↓
TTA 3,165 / ATA 1,303
        ↓
ATA 제외
        ↓
MusicGen TTA 동일 파일 충돌 3행 제외
        ↓
Clean TTA = 3,162
        ↓
12 generators
296 original_audio groups
Missing audio = 0
```

현재 연구의 FAKE 데이터는 **Clean TTA 3,162개**를 기준으로 진행한다.

---

## 9. 다음 단계

다음 노트북:

**`02_fma_matching_check.ipynb`**

목표:
1. Echoes Clean TTA에서 `original_audio` 296개 추출
2. FMA `track title + artist name`과 exact matching
3. 1개 일치 / 복수 후보 / 무매칭 구분
4. 장르·라이선스·track_id 규칙으로 후보 선택
5. `fma_real_mapping.csv` 생성
6. REAL 296곡 확보
